# NSC Bootstrap Stability Analysis — Pareto-Optimal Treatment Signatures

Identifies genes that reliably distinguish Pareto-optimal treatments from control using
Nearest Shrunken Centroid (NSC) bootstrap stability analysis on a Google Cloud Dataproc cluster.

# 1. Setup

In [1]:
import pandas as pd
import numpy as np
from sklearn.neighbors import NearestCentroid
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from scipy.cluster.hierarchy import linkage, leaves_list
from scipy.stats import ttest_ind
import plotly.express as px
import plotly.graph_objects as go
from pyspark.sql import SparkSession

_gex = pd.read_parquet('gs://gene_datasets/gene_expression_matrix.parquet', columns=['gene_id', 'gene_name'])
gene_symbol = dict(zip(_gex['gene_id'], _gex['gene_name']))

spark = SparkSession.builder.appName("NSC_bootstrap").getOrCreate()
sc = spark.sparkContext
print(f"Spark parallelism: {sc.defaultParallelism}")

# Load CPM matrix (samples × genes)
cpm = pd.read_parquet('gs://gene_datasets/sample_by_gene.parquet')

# Load PCA summary and identify Pareto-optimal treatments
pca_summary = pd.read_csv('gs://gene_datasets/pca_summary.csv')
pareto_treatments = pca_summary.loc[pca_summary['pareto_optimal'], 'sample_type'].tolist()
print(f"Pareto-optimal treatments ({len(pareto_treatments)}): {pareto_treatments}")

# Load Open Targets disease associations
disease_associations = pd.read_parquet('gs://gene_datasets/disease_associations.parquet')
clinically_relevant_genes = set(disease_associations['gene_id'].unique())
print(f"Open Targets genes: {len(clinically_relevant_genes)}")

26/06/03 18:28:32 WARN SparkSession: Using an existing Spark session; only runtime SQL configurations will take effect.


Spark parallelism: 2


/opt/conda/miniconda3/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.8) which Google will stop supporting in new releases of google.cloud._storage_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud._storage_v2 past that date.
  warnings.warn(message, FutureWarning)
/opt/conda/miniconda3/lib/python3.10/site-packages/google/api_core/_python_version_support.py:275: FutureWarning: You are using a Python version (3.10.8) which Google will stop supporting in new releases of google.cloud.storage_control_v2 once it reaches its end of life (2026-10-04). Please upgrade to the latest Python version, or at least Python 3.11, to continue receiving updates for google.cloud.storage_control_v2 past that date.
  warnings.warn(message, FutureWarning)


Pareto-optimal treatments (4): ['hATF567', 'hATF561', 'nZF105', 'nZF139']
Open Targets genes: 18845


# 2. Data Preparation

In [2]:
# Filter to Pareto-optimal treatments + Control
mask = cpm['sample_type'].isin(pareto_treatments + ['Control'])
sub = cpm[mask].copy()
gene_cols_all = [c for c in sub.columns if c.startswith('ENSG')]

print(f"Samples: {len(sub)}, Genes: {len(gene_cols_all)}")
print(sub['sample_type'].value_counts())

# Filter genes to Open Targets known set
gene_cols = [g for g in gene_cols_all if g in clinically_relevant_genes]
print(f"\nFiltered from {len(gene_cols_all)} to {len(gene_cols)} Open Targets genes")

# Scale and broadcast X, y
X_scaled = StandardScaler().fit_transform(sub[gene_cols].values)
y = (sub['sample_type'] != 'Control').astype(int).values

assert (y == 0).any() and (y == 1).any(), "y must contain both classes"

X_broadcast = sc.broadcast(X_scaled)
y_broadcast = sc.broadcast(y)
print("X and y broadcast to cluster")

Samples: 10, Genes: 78893
sample_type
hATF561     2
hATF567     2
nZF105      2
nZF139      2
Control     2
Base        0
nZF153      0
nZF93       0
nZF81       0
nZF42       0
nZF36       0
nZF156      0
nZF154      0
nZF147      0
nZF151      0
nZF148      0
nZF145      0
hATF555R    0
hATF555Q    0
ZDS2        0
SP1R        0
nZFD96      0
Name: count, dtype: int64

Filtered from 78893 to 18845 Open Targets genes
X and y broadcast to cluster


# 3. NSC Bootstrap Stability Analysis

For each bootstrap replicate, resample treatment and control indices with replacement and fit a
Nearest Shrunken Centroid classifier. A gene is "selected" when its class-centroid deviation
survives the shrinkage threshold. Selection frequency across 50 bootstraps measures stability.

Thresholds [0.5 → 3.0] are run in one Spark job (embarrassingly parallel across seed × threshold).

In [3]:
def fit_one_nsc_bootstrap_with_threshold(seed_and_threshold):
    import numpy as np
    from sklearn.neighbors import NearestCentroid

    seed, threshold = seed_and_threshold
    X, y_local = X_broadcast.value, y_broadcast.value
    rng = np.random.RandomState(seed)
    treat_idx = np.where(y_local == 1)[0]
    control_idx = np.where(y_local == 0)[0]
    idx = np.concatenate([
        rng.choice(treat_idx, len(treat_idx), replace=True),
        rng.choice(control_idx, len(control_idx), replace=True),
    ])
    nsc = NearestCentroid(shrink_threshold=threshold)
    nsc.fit(X[idx], y_local[idx])
    diff = nsc.centroids_[1] - nsc.centroids_[0]
    return (threshold, (diff != 0).astype(int).tolist())


THRESHOLDS = [0.5, 1.0, 1.5, 2.0, 2.5, 3.0]
N_BOOTSTRAP = 50

all_jobs = [(seed, t) for t in THRESHOLDS for seed in range(N_BOOTSTRAP)]
all_results = sc.parallelize(all_jobs, numSlices=20).map(fit_one_nsc_bootstrap_with_threshold).collect()

# Aggregate per threshold
threshold_stable_sets = {}
threshold_stability_dfs = {}

for threshold in THRESHOLDS:
    selections = [r[1] for r in all_results if r[0] == threshold]
    freq = np.array(selections).sum(axis=0) / N_BOOTSTRAP
    df = pd.DataFrame({'gene_id': gene_cols, 'selection_frequency': freq})
    threshold_stability_dfs[threshold] = df
    threshold_stable_sets[threshold] = set(df.query('selection_frequency == 1.0')['gene_id'])

    n100 = (freq == 1.0).sum()
    n70  = (freq >= 0.7).sum()
    n50  = (freq >= 0.5).sum()
    print(f"Threshold {threshold}: {n100} at 100%, {n70} at >=70%, {n50} at >=50%")

Threshold 0.5: 1484 at 100%, 5389 at >=70%, 9499 at >=50%
Threshold 1.0: 392 at 100%, 1640 at >=70%, 2863 at >=50%
Threshold 1.5: 96 at 100%, 308 at >=70%, 598 at >=50%
Threshold 2.0: 18 at 100%, 51 at >=70%, 89 at >=50%
Threshold 2.5: 7 at 100%, 7 at >=70%, 14 at >=50%
Threshold 3.0: 0 at 100%, 5 at >=70%, 7 at >=50%


# 4. Permutation Null Testing

Shuffle labels 100× per threshold to build a null distribution of gene counts.
P-value ≈ fraction of permutations that selected ≥ as many genes as the real data.

In [4]:
def fit_one_nsc_permuted(args):
    import numpy as np
    from sklearn.neighbors import NearestCentroid

    seed, threshold = args
    X = X_broadcast.value
    rng = np.random.RandomState(seed)
    y_permuted = rng.permutation(y_broadcast.value)
    nsc = NearestCentroid(shrink_threshold=threshold)
    nsc.fit(X, y_permuted)
    diff = nsc.centroids_[1] - nsc.centroids_[0]
    return (threshold, int((diff != 0).sum()))


N_PERMUTATIONS = 100
perm_jobs = [(seed, t) for t in THRESHOLDS for seed in range(N_PERMUTATIONS)]
perm_results = sc.parallelize(perm_jobs, numSlices=20).map(fit_one_nsc_permuted).collect()

print(f"\n{'Threshold':<10} {'Real (>=50%)':<16} {'Null mean ± std':<22} {'P-value'}")
for threshold in THRESHOLDS:
    perm_counts = [r[1] for r in perm_results if r[0] == threshold]
    real_count  = (threshold_stability_dfs[threshold]['selection_frequency'] >= 0.5).sum()
    perm_mean   = np.mean(perm_counts)
    perm_std    = np.std(perm_counts)
    p           = np.mean([c >= real_count for c in perm_counts])
    print(f"{threshold:<10} {real_count:<16} {perm_mean:.0f} ± {perm_std:.0f}{'':<12} {p:.3f}")


Threshold  Real (>=50%)     Null mean ± std        P-value
0.5        9499             5565 ± 821             0.000
1.0        2863             1045 ± 367             0.000
1.5        598              79 ± 51             0.000
2.0        89               12 ± 5             0.000
2.5        14               8 ± 3             0.030
3.0        7                0 ± 0             0.000


# 5. Threshold Refinement (2.0 – 2.5)

Re-run bootstrap + permutation at 0.1 increments to find the most stringent threshold
that remains significant (p < 0.001) against the null.

In [5]:
REFINE_THRESHOLDS = [2.0, 2.1, 2.2, 2.3, 2.4, 2.5]
N_BOOTSTRAP = 50
N_PERMUTATIONS = 100

# Bootstrap stability at refined thresholds
boot_jobs = [(seed, t) for t in REFINE_THRESHOLDS for seed in range(N_BOOTSTRAP)]
boot_results = sc.parallelize(boot_jobs, numSlices=20).map(fit_one_nsc_bootstrap_with_threshold).collect()

# Permutation null at refined thresholds
perm_jobs = [(seed, t) for t in REFINE_THRESHOLDS for seed in range(N_PERMUTATIONS)]
perm_results = sc.parallelize(perm_jobs, numSlices=20).map(fit_one_nsc_permuted).collect()

# Build stability dfs and print significance table
real_counts = {}
print(f"\n{'Threshold':<12} {'Real (>=50%)':<22} {'Null mean ± std':<22} {'P-value'}")
for threshold in REFINE_THRESHOLDS:
    selections = [r[1] for r in boot_results if r[0] == threshold]
    freq = np.array(selections).sum(axis=0) / N_BOOTSTRAP
    threshold_stability_dfs[threshold] = pd.DataFrame({'gene_id': gene_cols, 'selection_frequency': freq})

    real_counts[threshold] = (freq >= 0.5).sum()
    perm_counts = [r[1] for r in perm_results if r[0] == threshold]
    perm_mean = np.mean(perm_counts)
    perm_std  = np.std(perm_counts)
    p = np.mean([c >= real_counts[threshold] for c in perm_counts])
    print(f"{threshold:<12} {real_counts[threshold]:<22} {perm_mean:.0f} ± {perm_std:.0f}{'':<14} {p:.3f}")


Threshold    Real (>=50%)           Null mean ± std        P-value
2.0          89                     12 ± 5               0.000
2.1          66                     10 ± 4               0.000
2.2          44                     8 ± 4               0.000
2.3          30                     8 ± 3               0.000
2.4          22                     8 ± 3               0.000
2.5          14                     8 ± 3               0.030


# 6. Robust Gene Set Identification

- **3 ultra-robust genes**: 100%-stable across all thresholds 0.5 – 2.5
- **19-gene set**: ≥50% stability at threshold 2.2 (most stringent p < 0.001 threshold)
- **89-gene set**: ≥50% stability at threshold 2.0 (broader view for sensitivity analysis)
- **1,484-gene broad signature**: 100%-stable at threshold 0.5

In [6]:
# 3 ultra-robust genes: 100%-stable across thresholds 0.5–2.5
ultra_robust_ids = sorted(
    set.intersection(*[threshold_stable_sets[t] for t in THRESHOLDS if t <= 2.5])
)
print(f"Ultra-robust genes (100%-stable at all thresholds ≤2.5): {len(ultra_robust_ids)}")
print(ultra_robust_ids)

# 19-gene set: ≥50% stability at threshold 2.4
PRIMARY_THRESHOLD = 2.4
stable_19 = (threshold_stability_dfs[PRIMARY_THRESHOLD]
             .query('selection_frequency >= 0.5')
             .sort_values('selection_frequency', ascending=False))
print(f"\nGenes at threshold 2.4 (≥50% stability): {len(stable_19)}")
print(stable_19.to_string(index=False))

# 89-gene set at threshold 2.0
stable_89 = (threshold_stability_dfs[2.0]
             .query('selection_frequency >= 0.5')
             .sort_values('selection_frequency', ascending=False))
print(f"\nGenes at threshold 2.0 (≥50% stability): {len(stable_89)}")

# Broad signature: 100%-stable at threshold 0.5
broad_sig_df = threshold_stability_dfs[0.5].query('selection_frequency == 1.0').copy()
broad_gene_ids = broad_sig_df['gene_id'].tolist()
print(f"\nBroad signature (100%-stable at threshold 0.5): {len(broad_gene_ids)} genes")

# Save all gene sets to GCS
stable_19.to_csv('gs://gene_datasets/nsc_stable_genes_t22.csv', index=False)
stable_89.to_csv('gs://gene_datasets/nsc_stable_genes_t20.csv', index=False)
broad_sig_df[['gene_id', 'selection_frequency']].to_csv('gs://gene_datasets/nsc_broad_signature.csv', index=False)
print("\nSaved gene sets to gs://gene_datasets/")

Ultra-robust genes (100%-stable at all thresholds ≤2.5): 7
['ENSG00000047597', 'ENSG00000064218', 'ENSG00000108576', 'ENSG00000115085', 'ENSG00000132446', 'ENSG00000134627', 'ENSG00000168135']

Genes at threshold 2.4 (≥50% stability): 22
        gene_id  selection_frequency
ENSG00000047597                 1.00
ENSG00000168135                 1.00
ENSG00000064218                 1.00
ENSG00000108576                 1.00
ENSG00000115085                 1.00
ENSG00000132446                 1.00
ENSG00000134627                 1.00
ENSG00000178295                 0.68
ENSG00000189350                 0.64
ENSG00000188817                 0.64
ENSG00000177669                 0.64
ENSG00000138758                 0.62
ENSG00000140386                 0.60
ENSG00000160551                 0.58
ENSG00000226174                 0.56
ENSG00000151338                 0.52
ENSG00000055732                 0.52
ENSG00000119004                 0.52
ENSG00000092140                 0.52
ENSG00000276368       

# 7. Visualizations

In [7]:
# PCA before/after — all Open Targets genes (scaled)
pca = PCA(n_components=2)
pcs = pca.fit_transform(X_scaled)
sub = sub.reset_index()

plot_df = pd.DataFrame({
    'PC1': pcs[:, 0],
    'PC2': pcs[:, 1],
    'sample': sub['sample'].values,
    'group': sub['sample_type'].values,
})

fig = px.scatter(
    plot_df, x='PC1', y='PC2', color='group', hover_name='sample',
    title=f'PCA of Pareto + Control Samples — PC1 ({pca.explained_variance_ratio_[0]:.1%}), PC2 ({pca.explained_variance_ratio_[1]:.1%})',
    width=900, height=600,
)
fig.show()

In [8]:
# Heatmap of NSC-stable genes (PRIMARY_THRESHOLD)
stable_gene_ids = (threshold_stability_dfs[PRIMARY_THRESHOLD]
                   .query('selection_frequency >= 0.5')
                   .sort_values('selection_frequency', ascending=False)['gene_id']
                   .tolist())

stable_col_idx = [gene_cols.index(g) for g in stable_gene_ids]
X_stable = X_scaled[:, stable_col_idx]

heatmap_df = pd.DataFrame(X_stable, index=sub['sample'].values, columns=stable_gene_ids).T
sample_order = sub.sort_values('sample_type')['sample'].tolist()
heatmap_df = heatmap_df[sample_order]

sample_type_map = sub.set_index('sample')['sample_type'].to_dict()
x_labels = [f"{s}<br><i>{sample_type_map[s]}</i>" for s in heatmap_df.columns]

fig = go.Figure(go.Heatmap(
    z=heatmap_df.values,
    x=x_labels,
    y=[gene_symbol.get(g, g) for g in heatmap_df.index],
    colorscale='RdBu_r',
    zmid=0,
    colorbar=dict(title='Standardized<br>expression'),
))
fig.update_layout(
    title=f'NSC-Selected Gene Expression (threshold={PRIMARY_THRESHOLD}, {len(stable_gene_ids)} genes)',
    xaxis_title='Sample',
    yaxis_title='Gene',
    xaxis=dict(tickangle=-45),
    height=700, width=950,
    template='plotly_white',
)
fig.show()

The values are standardized expression (z-scores across samples, per gene). So for each gene:

Red = expression in that sample is above the gene's average across all samples
Blue = expression in that sample is below the gene's average across all samples
White = expression is right at the gene's average

## Broad Signature: 1,484-gene NSC set (threshold=0.5, 100% bootstrap stability)

In [9]:
# Broad-signature setup: extract 1,484-gene set and compute log2FC
broad_col_idx = [gene_cols.index(g) for g in broad_gene_ids]
X_broad = X_scaled[:, broad_col_idx]

X_broad_raw = sub[broad_gene_ids].values.astype(float)
is_control = (sub['sample_type'] == 'Control').values
is_treatment = ~is_control

ctrl_mean = X_broad_raw[is_control].mean(axis=0) + 0.5
treat_mean = X_broad_raw[is_treatment].mean(axis=0) + 0.5
log2fc = np.log2(treat_mean / ctrl_mean)

log2fc_df = pd.DataFrame({'gene_id': broad_gene_ids, 'log2FC': log2fc})
print(f"Broad signature: {len(broad_gene_ids)} genes")
print(f"log2FC range: [{log2fc.min():.2f}, {log2fc.max():.2f}]")
print(f"Upregulated: {(log2fc > 0).sum()}, Downregulated: {(log2fc < 0).sum()}")

Broad signature: 1484 genes
log2FC range: [-0.88, 1.45]
Upregulated: 483, Downregulated: 1001


In [10]:
# Broad-signature clustered heatmap (Ward linkage on 1,484 genes × 10 samples)
sample_order_df = sub[['sample', 'sample_type']].copy()
sample_order_df['_ctrl'] = (sample_order_df['sample_type'] == 'Control').astype(int)
sample_order_df = sample_order_df.sort_values(['_ctrl', 'sample_type'], ascending=[False, True])
ordered_samples = sample_order_df['sample'].tolist()

heatmap_broad = pd.DataFrame(
    X_broad, index=sub['sample'].values, columns=broad_gene_ids,
).T[ordered_samples]

Z = linkage(heatmap_broad.values, method='ward', metric='euclidean')
row_order = leaves_list(Z)
heatmap_sorted = heatmap_broad.iloc[row_order]

n_genes = len(heatmap_sorted)
y_tickvals = list(range(0, n_genes, 100))
x_labels = [f"{s}<br><i>{sample_type_map[s]}</i>" for s in ordered_samples]

fig = go.Figure(go.Heatmap(
    z=heatmap_sorted.values,
    x=x_labels,
    y=list(range(n_genes)),
    colorscale='RdBu_r',
    zmid=0, zmin=-3, zmax=3,
    colorbar=dict(title='Standardized<br>expression (z-score)', thickness=18),
))
fig.update_layout(
    title=dict(
        text=('Clustered Expression Heatmap — 1,484-Gene NSC Broad Signature<br>'
              '<sup>Rows: genes by Ward linkage | Columns: samples, Controls grouped left</sup>'),
        x=0.5,
    ),
    xaxis_title='Sample',
    yaxis=dict(
        title='Gene rank (hierarchical order)',
        tickmode='array', tickvals=y_tickvals,
        ticktext=[str(i) for i in y_tickvals],
    ),
    xaxis=dict(tickangle=-35),
    height=800, width=950,
    template='plotly_white',
)
fig.show()

In [11]:
# Volcano plot — 1,484 stable genes in context of all 18,845 Open Targets genes
all_raw = sub[gene_cols].values.astype(float)
ctrl_expr  = all_raw[is_control]
treat_expr = all_raw[is_treatment]

ctrl_mean_all  = ctrl_expr.mean(axis=0) + 0.5
treat_mean_all = treat_expr.mean(axis=0) + 0.5
log2fc_all = np.log2(treat_mean_all / ctrl_mean_all)

t_stat, p_val = ttest_ind(treat_expr, ctrl_expr, axis=0, equal_var=False)
p_val = np.where(p_val == 0, 1e-300, p_val)
neg_log10_p = -np.log10(p_val)

all_gene_stats = pd.DataFrame({
    'gene_id': gene_cols,
    'log2FC': log2fc_all,
    't_stat': t_stat,
    'p_value': p_val,
    'neg_log10_p': neg_log10_p,
})

is_stable_mask = np.isin(gene_cols, broad_gene_ids)
gray_df   = all_gene_stats[~is_stable_mask]
stable_df = all_gene_stats[is_stable_mask]

fig = go.Figure()
fig.add_trace(go.Scattergl(
    x=gray_df['log2FC'], y=gray_df['neg_log10_p'],
    mode='markers',
    marker=dict(color='lightgray', size=3, opacity=0.5),
    name=f'Other genes (n={len(gray_df):,})',
    hovertemplate='%{customdata}<br>log2FC=%{x:.2f}<br>-log10(p)=%{y:.2f}<extra></extra>',
    customdata=gray_df['gene_id'].map(lambda g: gene_symbol.get(g, g)),
))
fig.add_trace(go.Scattergl(
    x=stable_df['log2FC'], y=stable_df['neg_log10_p'],
    mode='markers',
    marker=dict(color='mediumpurple', size=5, opacity=0.8),
    name=f'NSC broad stable (n={len(stable_df):,})',
    hovertemplate='%{customdata}<br>log2FC=%{x:.2f}<br>-log10(p)=%{y:.2f}<extra></extra>',
    customdata=stable_df['gene_id'].map(lambda g: gene_symbol.get(g, g)),
))
fig.add_vline(x=0, line_width=1, line_dash='dash', line_color='black')
fig.update_layout(
    title=dict(
        text=('Volcano Plot — NSC Broad Signature (1,484 genes) in Context<br>'
              "<sup>Treatment vs Control | Welch's t-test | Purple = 100%-stable at threshold=0.5</sup>"),
        x=0.5,
    ),
    xaxis_title='log₂ Fold Change (treatment / control)',
    yaxis_title='−log₁₀(p-value)',
    legend=dict(x=0.02, y=0.98, bgcolor='rgba(255,255,255,0.8)'),
    template='plotly_white',
    width=950, height=600,
)
fig.show()

# 8. Open Targets Enrichment of Stable Genes

In [ ]:
stable_genes = threshold_stability_dfs[PRIMARY_THRESHOLD].query('selection_frequency >= 0.5')

nsc_enriched = stable_genes.merge(disease_associations, on='gene_id', how='left')
print(f"Stable genes: {len(stable_genes)}")
print(f"Genes with ≥1 disease association: {nsc_enriched.dropna(subset=['disease_name'])['gene_id'].nunique()}")

# Top diseases linked to stable genes (≥2 genes converging on the same trait)
top_diseases = (nsc_enriched
                .dropna(subset=['disease_name'])
                .groupby('disease_name')
                .agg(n_genes=('gene_id', 'nunique'),
                     avg_score=('association_score', 'mean'),
                     max_score=('association_score', 'max'))
                .query('n_genes >= 2')
                .sort_values(['n_genes', 'avg_score'], ascending=False)
                .head(20))

# Therapeutic area summary
exploded = (nsc_enriched
            .dropna(subset=['disease_name'])
            .explode('therapeutic_areas')
            .dropna(subset=['therapeutic_areas']))

top_areas = (exploded
             .groupby('therapeutic_areas')
             .agg(n_genes=('gene_id', 'nunique'),
                  n_diseases=('disease_name', 'nunique'),
                  avg_score=('association_score', 'mean'))
             .sort_values('n_genes', ascending=False))

# Ultra-robust gene deep-dive
ultra_robust_details = (nsc_enriched[nsc_enriched['gene_id'].isin(ultra_robust_ids)]
                        .dropna(subset=['disease_name'])
                        .sort_values(['gene_id', 'association_score'], ascending=[True, False]))

print("\nTop diseases (≥2 stable genes):")
display(top_diseases)

print("\nTherapeutic areas:")
display(top_areas)

print("\n3 ultra-robust genes — top associations:")
print(ultra_robust_details.head(30).to_string(index=False))

Stable genes: 22
Genes with ≥1 disease association: 22

Top diseases (≥2 stable genes):


,n_genes,avg_score,max_score
disease_name,,,
neurodegenerative disease,10,0.364820,0.556603
body height,8,0.317413,0.511564
platelet volume,7,0.357856,0.521286
erythrocyte count,6,0.370091,0.506992
body mass index,6,0.362208,0.514759
sitting height measurement,6,0.348904,0.499292
eosinophil count,6,0.338996,0.538651
serum creatinine amount,6,0.297800,0.475605
Red cell distribution width,6,0.285197,0.355384



Therapeutic areas:


,n_genes,n_diseases,avg_score
therapeutic_areas,,,
measurement,19,207,0.280034
nervous system disease,18,70,0.353967
phenotype,13,69,0.331464
"genetic, familial or congenital disease",12,40,0.348303
reproductive system or breast disease,11,17,0.239081
cancer or benign tumor,11,21,0.231605
musculoskeletal or connective tissue disease,10,21,0.259994
psychiatric disorder,8,37,0.372303
gastrointestinal disease,8,17,0.261097



3 ultra-robust genes — top associations:
        gene_id  selection_frequency                                              disease_name    disease_id                                                                                                          therapeutic_area_ids  association_score                                                                                               therapeutic_areas
ENSG00000047597                  1.0                       McLeod neuroacanthocytosis syndrome MONDO_0018945 {'list': [{'element': 'EFO_0000618'}, {'element': 'OTAR_0000018'}, {'element': 'MONDO_0002025'}, {'element': 'EFO_0000651'}]}           0.804145              [nervous system disease, genetic, familial or congenital disease, psychiatric disorder, phenotype]
ENSG00000047597                  1.0                                          genetic disorder   EFO_0000508                                                                                       {'list': [{'element': 'OTAR_000

In [13]:
# Open Targets therapeutic area enrichment — 1,484-gene broad signature
broad_disease = disease_associations[disease_associations['gene_id'].isin(set(broad_gene_ids))].copy()
print(f"Broad-signature genes with OT associations: {broad_disease['gene_id'].nunique()} / {len(broad_gene_ids)}")

broad_ta = (broad_disease
            .dropna(subset=['therapeutic_areas'])
            .explode('therapeutic_areas')
            .dropna(subset=['therapeutic_areas']))

ta_counts = (broad_ta
             .groupby('therapeutic_areas')['gene_id']
             .nunique()
             .reset_index()
             .rename(columns={'gene_id': 'n_genes'})
             .sort_values('n_genes', ascending=True))

fig = go.Figure(go.Bar(
    x=ta_counts['n_genes'],
    y=ta_counts['therapeutic_areas'],
    orientation='h',
    marker_color='mediumslateblue',
    text=ta_counts['n_genes'],
    textposition='outside',
))
fig.update_layout(
    title=dict(
        text=('Open Targets Therapeutic Area Enrichment — 1,484-Gene NSC Broad Signature<br>'
              '<sup>Count of unique stable genes per therapeutic area</sup>'),
        x=0.5,
    ),
    xaxis_title='Number of stable genes',
    yaxis_title='Therapeutic area',
    template='plotly_white',
    height=max(500, 28 * len(ta_counts)),
    width=950,
    margin=dict(l=280),
)
fig.show()

Broad-signature genes with OT associations: 1484 / 1484


In [14]:
# # Save enrichment results to GCS
# top_diseases.to_csv('gs://gene_datasets/nsc_top_diseases.csv')
# top_areas.to_csv('gs://gene_datasets/nsc_top_areas.csv')
# log2fc_df.to_csv('gs://gene_datasets/nsc_broad_log2fc.csv', index=False)
# all_gene_stats.to_csv('gs://gene_datasets/nsc_all_gene_stats.csv', index=False)
# print("Saved enrichment results to gs://gene_datasets/")

In [15]:
# Build long-format per-gene log2FC table for each Pareto-optimal treatment

gene_cols = [c for c in sub.columns if c.startswith('ENSG')]
control_means = sub.loc[sub['sample_type'] == 'Control', gene_cols].mean(axis=0)
pseudocount = 0.5

log2fc_records = []
for tx in pareto_treatments:
    tx_means = sub.loc[sub['sample_type'] == tx, gene_cols].mean(axis=0)
    log2fc = np.log2((tx_means + pseudocount) / (control_means + pseudocount))
    
    log2fc_records.append(pd.DataFrame({
        'treatment': tx,
        'gene_id': gene_cols,
        'log2fc': log2fc.values,
        'abs_log2fc': np.abs(log2fc.values),
    }))

log2fc_long = pd.concat(log2fc_records, ignore_index=True)
print(f"log2fc_long shape: {log2fc_long.shape}")
print(log2fc_long.head())

log2fc_long shape: (315572, 4)
  treatment          gene_id    log2fc  abs_log2fc
0   hATF567  ENSG00000000003 -0.250679    0.250679
1   hATF567  ENSG00000000005  0.000000    0.000000
2   hATF567  ENSG00000000419 -0.052726    0.052726
3   hATF567  ENSG00000000457 -0.091690    0.091690
4   hATF567  ENSG00000000460 -0.528262    0.528262


In [16]:
# Reuse: log2fc_long (per (treatment, gene) log2FC values), disease_associations, pareto_treatments
# log2fc_long should have columns: treatment, gene_id, log2fc, abs_log2fc

risk_metrics = []

for tx in pareto_treatments:
    # Step 1: Get this treatment's meaningfully disturbed genes
    tx_disturbed = log2fc_long.query("treatment == @tx and abs_log2fc >= 1.0")
    
    # Step 2: Join with disease associations to get gene-disease evidence
    disturbed_with_disease = tx_disturbed.merge(
        disease_associations, on='gene_id', how='inner'
    )
    
    # Step 3: Compute per-treatment metrics
    metrics = {
        'treatment': tx,
        
        # A. Breadth — perturbation footprint size
        'n_disturbed_genes': tx_disturbed['gene_id'].nunique(),
        'n_disease_linked_disturbed_genes': disturbed_with_disease['gene_id'].nunique(),
        
        # B. Weighted evidence — sum of (effect magnitude × evidence strength)
        # This is the key metric incorporating association_score
        'evidence_weighted_burden': (
            disturbed_with_disease['abs_log2fc'] * disturbed_with_disease['association_score']
        ).sum(),
        
        # C. High-confidence perturbations — both strong effect AND strong evidence
        'n_high_confidence_perturbations': len(
            disturbed_with_disease.query('abs_log2fc >= 1.5 and association_score >= 0.5')
            .drop_duplicates('gene_id')
        ),
        
        # D. Coverage — distinct therapeutic areas affected
        'n_therapeutic_areas_touched': (
            disturbed_with_disease.dropna(subset=['therapeutic_areas'])
            .explode('therapeutic_areas')
            ['therapeutic_areas'].nunique()
        ),
    }
    risk_metrics.append(metrics)

risk_df = pd.DataFrame(risk_metrics)
print("Per-treatment off-target risk metrics:")
print(risk_df.to_string(index=False))

# Step 4: Compute composite rank
# Rank each metric (higher rank = higher risk)
for col in ['n_disturbed_genes', 'evidence_weighted_burden',
            'n_high_confidence_perturbations', 'n_therapeutic_areas_touched']:
    risk_df[f'rank_{col}'] = risk_df[col].rank()

# Composite = average of ranks (equally weighted)
rank_cols = [c for c in risk_df.columns if c.startswith('rank_')]
risk_df['composite_risk_rank'] = risk_df[rank_cols].mean(axis=1)

risk_df = risk_df.sort_values('composite_risk_rank', ascending=False)

print("\nFinal off-target risk ranking (highest to lowest):")
print(risk_df[['treatment', 'n_disturbed_genes', 'evidence_weighted_burden',
               'n_high_confidence_perturbations', 'n_therapeutic_areas_touched',
               'composite_risk_rank']].round(2).to_string(index=False))

# Save
#risk_df.to_csv('gs://gene_datasets/treatment_risk_scores.csv', index=False)

Per-treatment off-target risk metrics:
treatment  n_disturbed_genes  n_disease_linked_disturbed_genes  evidence_weighted_burden  n_high_confidence_perturbations  n_therapeutic_areas_touched
  hATF567                 62                                17                236.982548                                1                           24
  hATF561                 25                                 6                 43.567403                                0                           20
   nZF105                 61                                18                130.408629                                0                           22
   nZF139                 64                                14                138.992137                                2                           22

Final off-target risk ranking (highest to lowest):
treatment  n_disturbed_genes  evidence_weighted_burden  n_high_confidence_perturbations  n_therapeutic_areas_touched  composite_risk_rank
  hATF567       

Running the above code with no cutoff creates too much noise. With all 80,000 genes included we have too much noise and there is not much variation between the treatments. When we reintroduce the |log2FC|>=1 cutoff we find now that the most risky treatment is hATF567, with the least risky treatment being hATF561. hATF561 is the least risky becuase it only disturbs 25 genes at this threshold, and 0 high confidence perteubations, with only 20 therapeutic areas and 6 disease linked genes. 

In [17]:
risk_df

,treatment,n_disturbed_genes,n_disease_linked_disturbed_genes,evidence_weighted_burden,n_high_confidence_perturbations,n_therapeutic_areas_touched,rank_n_disturbed_genes,rank_evidence_weighted_burden,rank_n_high_confidence_perturbations,rank_n_therapeutic_areas_touched,composite_risk_rank
0,hATF567,62,17,236.982548,1,24,3.0,4.0,3.0,4.0,3.500
3,nZF139,64,14,138.992137,2,22,4.0,3.0,4.0,2.5,3.375
2,nZF105,61,18,130.408629,0,22,2.0,2.0,1.5,2.5,2.000
1,hATF561,25,6,43.567403,0,20,1.0,1.0,1.5,1.0,1.125


In [18]:
metrics_to_plot = ['n_disturbed_genes', 'evidence_weighted_burden',
                   'n_high_confidence_perturbations', 'n_therapeutic_areas_touched']
metric_labels = ['Disturbed<br>genes', 'Evidence-weighted<br>burden',
                 'High-confidence<br>perturbations', 'Therapeutic areas<br>touched']

# Sort by composite rank (highest risk on top)
risk_df_sorted = risk_df.sort_values('composite_risk_rank', ascending=False)

# Normalize for color scale
normalized = risk_df_sorted[metrics_to_plot].copy()
for col in metrics_to_plot:
    col_min, col_max = normalized[col].min(), normalized[col].max()
    normalized[col] = (normalized[col] - col_min) / (col_max - col_min + 1e-10)

# Display values as actual numbers
display_values = risk_df_sorted[metrics_to_plot].round(1).astype(str).values

fig = go.Figure(go.Heatmap(
    z=normalized.values,
    x=metric_labels,
    y=risk_df_sorted['treatment'].values,
    colorscale='Reds',
    text=display_values,
    texttemplate='%{text}',
    textfont=dict(size=13),
    colorbar=dict(title='Normalized<br>risk score'),
    hovertemplate='Treatment: %{y}<br>Metric: %{x}<br>Normalized: %{z:.2f}<extra></extra>',
))

fig.update_layout(
    title='Off-Target Risk Profile by Treatment<br><sub>Treatments ranked highest to lowest composite risk</sub>',
    height=400, width=850,
    template='plotly_white',
    yaxis=dict(autorange='reversed'),  # ensures highest risk is on top
)
fig.show()

In [ ]:
# Decompose evidence-weighted burden into shared (NSC-selected) vs treatment-specific components
stable_genes_t24 = (threshold_stability_dfs[2.4]
                    .query('selection_frequency >= 0.5')['gene_id'].tolist())
stable_genes_t05 = (threshold_stability_dfs[0.5]
                    .query('selection_frequency == 1.0')['gene_id'].tolist())

nsc_set = set(stable_genes_t05)  # use the 1,484-gene broad signature

split_rows = []
for tx in pareto_treatments:
    tx_data = (log2fc_long.query("treatment == @tx and abs_log2fc >= 1.0")
               .merge(disease_associations, on='gene_id', how='inner'))
    
    shared = tx_data[tx_data['gene_id'].isin(nsc_set)]
    specific = tx_data[~tx_data['gene_id'].isin(nsc_set)]
    
    split_rows.append({
        'treatment': tx,
        'n_shared_genes': shared['gene_id'].nunique(),
        'shared_burden': (shared['abs_log2fc'] * shared['association_score']).sum(),
        'n_specific_genes': specific['gene_id'].nunique(),
        'specific_burden': (specific['abs_log2fc'] * specific['association_score']).sum(),
    })

split_df = pd.DataFrame(split_rows)
split_df['total_burden'] = split_df['shared_burden'] + split_df['specific_burden']
split_df['pct_shared'] = split_df['shared_burden'] / split_df['total_burden'] * 100
split_df = split_df.sort_values('total_burden', ascending=False)
print(split_df.round(2).to_string(index=False))

#split_df.to_csv('gs://gene_datasets/treatment_burden_split.csv', index=False)

treatment  n_shared_genes  shared_burden  n_specific_genes  specific_burden  total_burden  pct_shared
  hATF567              12         134.16                 5           102.82        236.98       56.61
   nZF139               8          24.25                 6           114.74        138.99       17.45
   nZF105              13          82.06                 5            48.35        130.41       62.93
  hATF561               1           1.01                 5            42.56         43.57        2.31


In [20]:
fig = go.Figure()
fig.add_trace(go.Bar(
    name='Shared (NSC-selected)',
    y=split_df['treatment'], x=split_df['shared_burden'],
    orientation='h', marker_color='#534AB7',
    text=split_df['shared_burden'].round(1), textposition='inside',
))
fig.add_trace(go.Bar(
    name='Treatment-specific',
    y=split_df['treatment'], x=split_df['specific_burden'],
    orientation='h', marker_color='#B4B2A9',
    text=split_df['specific_burden'].round(1), textposition='inside',
))
fig.update_layout(
    barmode='stack',
    title='Evidence-Weighted Burden: Shared vs Treatment-Specific Components',
    xaxis_title='Evidence-weighted burden',
    yaxis=dict(autorange='reversed'),
    height=400, width=900,
    template='plotly_white',
    legend=dict(orientation='h', y=-0.15),
)
fig.show()

I filtered to genes with |log2FC| ≥ 1.0 that also had Open Targets disease associations, then split those into NSC-selected genes (shared across treatments) vs treatment-specific genes, and computed the evidence-weighted burden separately for each component

hATF561 (2.31%) — almost its entire burden is treatment-specific. Its one shared gene contributes almost nothing. This treatment is doing something idiosyncratic that the NSC didn't capture as a general pattern. Combined with it having the lowest total burden, this is actually the cleanest profile: low burden, and what burden exists isn't a systematic off-target effect.
nZF139 (17.45%) — also mostly specific burden, but total burden is high (~139). The specific component (114.74) is doing most of the work. This treatment is hitting disease-linked genes that other treatments aren't — that's the more concerning safety signal.
nZF105 (62.93%) and hATF567 (56.61%) — majority of their burden is shared. This means a large fraction of their off-target footprint is hitting genes that the NSC flagged as systematically perturbed across the panel. That's arguably less concerning than specific burden — it reflects a class effect of zinc finger binding rather than idiosyncratic disruption — but the total burdens are also the two highest.